# Applied AI: Vision, Generative, RL, RecSys, Time Series, Anomalies, Fairness

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/14_applied_ai.ipynb)

Runnable examples from **Data/ML Handbook** — section `14_applied_ai`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/14_applied_ai/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s11_timeseries.py`

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb
rng = np.random.default_rng(0); idx = pd.date_range("2024-01-01", periods=730, freq="D")
y = pd.Series(100 + 0.05 * np.arange(730) + 15 * np.sin(2 * np.pi * idx.dayofweek / 7) + 10 * np.sin(2 * np.pi * idx.dayofyear / 365) + rng.normal(0, 4, 730), idx)
df = pd.DataFrame({"y": y}); H = 28
for L in [H, H + 7, 364]: df[f"lag_{L}"] = df.y.shift(L)              # lags >= horizon: no leakage
df["roll_mean_28"] = df.y.shift(H).rolling(28).mean(); df["dow"] = idx.dayofweek; df["doy"] = idx.dayofyear
train, test = df.iloc[:-H].dropna(), df.iloc[-H:]
m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, verbose=-1).fit(train.drop(columns="y"), train.y)
pred = m.predict(test.drop(columns="y"))
X_tr, X_te = train.drop(columns="y"), test.drop(columns="y")
m2 = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, verbose=-1).fit(X_tr, train.y - train.roll_mean_28)
pred2 = m2.predict(X_te) + test.roll_mean_28                             # trees can't extrapolate: model the residual
snaive = df.y.shift(H).iloc[-H:].values                                  # naive: value 28 days earlier (same weekday)
mae = lambda a, b: np.mean(np.abs(np.asarray(a) - np.asarray(b)))
scale = mae(train.y.values[7:], train.y.values[:-7])                     # in-sample seasonal-naive MAE (lag 7)
for name, p in [("seasonal naive (t-28)", snaive), ("LightGBM on raw y", pred), ("LightGBM on y - level", pred2)]:
    print(f"{name:24s} MAE={mae(test.y, p):5.2f}  MASE={mae(test.y, p)/scale:.2f}")

### `code/s11_anomaly_rec_rl.py`

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
rng = np.random.default_rng(0)
X = np.r_[rng.normal(0, 1, (2000, 5)), rng.normal(0, 1, (20, 5)) + rng.choice([-4, 4], (20, 5))]; lab = np.r_[np.zeros(2000), np.ones(20)]
iso = IsolationForest(n_estimators=300, contamination="auto", random_state=0).fit(X)
print(f"IsolationForest PR-AUC={average_precision_score(lab, -iso.score_samples(X)):.3f} (baseline {lab.mean():.3f})")
# matrix factorisation by ALS on a toy ratings matrix
R = np.array([[5, 4, 0, 1], [4, 0, 0, 1], [1, 1, 0, 5], [0, 1, 5, 4], [1, 0, 4, 0]], float); M = R > 0
k, lam = 2, 0.1; U, V = rng.normal(0, .1, (5, k)), rng.normal(0, .1, (4, k))
for _ in range(30):
    for u in range(5): U[u] = np.linalg.solve(V[M[u]].T @ V[M[u]] + lam * np.eye(k), V[M[u]].T @ R[u, M[u]])
    for i in range(4): V[i] = np.linalg.solve(U[M[:, i]].T @ U[M[:, i]] + lam * np.eye(k), U[M[:, i]].T @ R[M[:, i], i])
print("ALS reconstruction (0 = unseen in input):\n", (U @ V.T).round(1))
# tabular Q-learning on a 1-D corridor: reward at the right end
nS, Q, alpha, gamma, eps = 6, np.zeros((6, 2)), 0.5, 0.9, 0.2
for ep in range(300):
    s = 0
    while s != nS - 1:
        a = rng.integers(2) if rng.random() < eps else int(Q[s].argmax())
        s2 = max(0, s - 1) if a == 0 else s + 1; r = 1.0 if s2 == nS - 1 else -0.01
        Q[s, a] += alpha * (r + gamma * Q[s2].max() * (s2 != nS - 1) - Q[s, a]); s = s2
print("greedy policy (1=right):", Q.argmax(1)[:-1], " V(s0)=", Q[0].max().round(3))

### `code/s11_fairness.py`

In [ ]:
import numpy as np
rng = np.random.default_rng(0); n = 10_000
g = rng.choice(["A", "B"], n, p=[.7, .3]); y = rng.random(n) < np.where(g == "A", .30, .20)
score = np.clip(y * .35 + rng.normal(.35, .18, n) + (g == "A") * .05, 0, 1); yhat = score > .5
for grp in ["A", "B"]:
    m = g == grp
    print(f"group {grp}: selection rate={yhat[m].mean():.3f}  TPR={yhat[m & y].mean():.3f}  FPR={yhat[m & ~y].mean():.3f}  precision={y[m & yhat].mean():.3f}")
sr = [yhat[g == k].mean() for k in "AB"]; print(f"disparate impact ratio = {min(sr)/max(sr):.2f}  (< 0.8 fails the 80% rule)")